In [1]:
import os
import time
import numpy as np
import jax
import jax.numpy as jnp
from jax import lax, jit, vmap
from jax.experimental import pjit
from jax.sharding import Mesh, PositionalSharding
from functools import partial

# --- Configuration (smaller scale for prototyping)
MAX_RECURSION_DEPTH    = 100_000      # Maximum recursion depth
TOTAL_DEPTH            = 100_000      # Total recursion depth (fused into one iteration)
OPTIMAL_DEPTH_STEP     = TOTAL_DEPTH  # Full fusion
DIMENSIONAL_CONSTRAINT = 0.8
BATCH_SIZE             = 1_000_000    # 1M samples (smaller batch for prototyping)

# Use up to 8 devices, or however many are available
NUM_DEVICES = min(8, jax.device_count())
LOCAL_BATCH_SIZE = BATCH_SIZE // NUM_DEVICES

VAL_CLAMP_LOW  = -100.0
VAL_CLAMP_HIGH =  100.0

# --- Setup Device Mesh for pjit
devices = jax.devices()[:NUM_DEVICES]
mesh = Mesh(devices, ("data",))
sharding = PositionalSharding(mesh.devices.flat)

# --- Data Lake Directory (to store processed output)
DATA_LAKE_DIR = "datalake"
if not os.path.exists(DATA_LAKE_DIR):
    os.makedirs(DATA_LAKE_DIR)

# --- Core Code Engine: Optimized Processing Functions
@jit
def dynamic_pi(depth, scale_factor):
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)
    return jnp.pi * jnp.log1p(depth + 1) * scale_factor * DIMENSIONAL_CONSTRAINT

@jit
def dynamic_phi(depth, scale_factor):
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)
    return (1 + jnp.sqrt(5)) / 2 * jnp.exp(-depth / ((scale_factor * 20) + 1)) * DIMENSIONAL_CONSTRAINT

@jit
def stabilize_depth(depth):
    return depth / (1 + jnp.log1p(depth + 1))

@partial(jit, static_argnames=["depth"])
def dppu_with_dynamic_pi_phi(x, depth=OPTIMAL_DEPTH_STEP, scale_factor=1.0):
    depth = stabilize_depth(jnp.minimum(depth, MAX_RECURSION_DEPTH))
    def body_fn(i, val):
        pi_dyn  = dynamic_pi(i, scale_factor)
        phi_dyn = dynamic_phi(i, scale_factor)
        scale   = jnp.log1p(i + 1) * scale_factor * DIMENSIONAL_CONSTRAINT
        safe_val = jnp.clip(val, VAL_CLAMP_LOW, VAL_CLAMP_HIGH)
        new_val = jnp.sin(safe_val * scale * pi_dyn) * jnp.exp(-safe_val / (phi_dyn + 10))
        return new_val
    return lax.fori_loop(0, depth.astype(jnp.int32), body_fn, x)

def branch_recycle(x, num_branches=2, branch_depth=OPTIMAL_DEPTH_STEP, scale_factor=1.0):
    xs = jnp.stack([x] * num_branches, axis=0)
    branch_fn = vmap(lambda xi: dppu_with_dynamic_pi_phi(xi, depth=branch_depth, scale_factor=scale_factor))
    branch_outputs = branch_fn(xs)
    return jnp.sum(branch_outputs, axis=0)

# --- Pipeline using pjit: full fusion (one iteration)
@partial(pjit.pjit,
         in_shardings=(sharding,),
         out_shardings=sharding,
         static_argnames=("num_branches", "branch_depth", "scale_factor"))
def process_full(x, num_branches, branch_depth, scale_factor):
    return branch_recycle(x, num_branches=num_branches, branch_depth=branch_depth, scale_factor=scale_factor)

def run_pipeline(batch_input, total_depth=TOTAL_DEPTH, num_branches=2,
                 branch_depth=OPTIMAL_DEPTH_STEP, scale_factor=1.0):
    # With full fusion, we have a single iteration.
    sharded_input = jax.device_put(batch_input, sharding)
    start_time = time.time()
    final_output = process_full(sharded_input, num_branches, branch_depth, scale_factor)
    final_output = jax.device_get(final_output)
    jax.block_until_ready(final_output)
    elapsed = time.time() - start_time
    mean_val = float(jnp.mean(final_output))
    return final_output, elapsed, mean_val

# --- Data Lake: Save Processed Data to Disk
def save_to_data_lake(data, filename="processed_data.npy"):
    filepath = os.path.join(DATA_LAKE_DIR, filename)
    np.save(filepath, np.array(data))
    print(f"Saved processed data to {filepath}")

def load_from_data_lake(filename="processed_data.npy"):
    filepath = os.path.join(DATA_LAKE_DIR, filename)
    if os.path.exists(filepath):
        data = np.load(filepath)
        print(f"Loaded processed data from {filepath}")
        return data
    else:
        print(f"No file found at {filepath}")
        return None

# --- Main: End-to-End Pipeline Prototype
if __name__ == "__main__":
    print("Generating raw data...")
    batch_input = jnp.linspace(0, 10, BATCH_SIZE)

    print("Running processing pipeline...")
    processed_data, elapsed, mean_val = run_pipeline(batch_input)
    print(f"Processing completed in {elapsed:.2f} sec with mean output {mean_val:.6f}")

    print("Saving processed data to Data Lake...")
    save_to_data_lake(processed_data, filename="processed_data.npy")

    # Optional: Load data back for debugging.
    loaded_data = load_from_data_lake("processed_data.npy")
    if loaded_data is not None:
        print(f"Loaded data shape: {loaded_data.shape}")






Generating raw data...
Running processing pipeline...
Processing completed in 213.76 sec with mean output -0.031019
Saving processed data to Data Lake...
Saved processed data to datalake/processed_data.npy
Loaded processed data from datalake/processed_data.npy
Loaded data shape: (1000000,)
